In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

In [2]:
DATASET_PATH = "../data/train.csv"

LOW_QUANTILE = 0.10
HIGH_QUANTILE = 0.90
IRQ_COEFF = 3

# Data Processing

### Loading Dataset

In [3]:
ds = pd.read_csv(DATASET_PATH)

important_df =  ds[['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'Fee', 'State', 'AdoptionSpeed']]

important_df = important_df.drop(columns=['Color3', 'Fee', 'State'])

In [4]:
important_df.head(5)

,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


### Handling NaN values

In [5]:
missing = list()
for x in important_df.columns:
    if important_df[x].isnull().sum() != 0:
        print(f"{x:<30}{important_df[x].isnull().sum():<10}{(important_df[x].isnull().sum() / important_df.shape[0])*100}%")
        missing.append(x)

In [6]:
important_df.head(5)

,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


### Detecting anomalies

In [7]:
def detect_anomalies(df, column):
    Q1 = df[column].quantile(LOW_QUANTILE)
    Q3 = df[column].quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR
    anomalies = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return anomalies

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    anomalies = detect_anomalies(important_df, col)
    print(f"Anomalies in {col}:")
    print(anomalies.shape[0])

Anomalies in Age:
62
Anomalies in Breed1:
0
Anomalies in Breed2:
0
Anomalies in Gender:
0
Anomalies in Color1:
0
Anomalies in Color2:
0
Anomalies in MaturitySize:
0
Anomalies in Vaccinated:
0
Anomalies in FurLength:
0
Anomalies in Dewormed:
0
Anomalies in Sterilized:
0
Anomalies in Health:
515
Anomalies in Quantity:
62
Anomalies in AdoptionSpeed:
0


In [8]:
def replace_anomalies_with_minmax_values(df, column):
    sorted_values = df[column].sort_values()
    Q1 = sorted_values.quantile(LOW_QUANTILE)
    Q3 = sorted_values.quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR

    min_real_value = sorted_values[sorted_values >= lower_bound].min()
    max_real_value = sorted_values[sorted_values <= upper_bound].max()

    df.loc[df[column] < lower_bound, column] = min_real_value
    df.loc[df[column] > upper_bound, column] = max_real_value

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    replace_anomalies_with_minmax_values(important_df, col)

print("Anomalies replaced with real values. Updated DataFrame:")
important_df.head(5)

def scale_columns(df, columns):
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()
        df[col] = (df[col] - mean) / std

if False:
    numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']
    
    scale_columns(important_df, numerical_columns)

    print("Numerical columns scaled using z-score normalization:")
    important_df.head(5)

Anomalies replaced with real values. Updated DataFrame:


# Hyperparams tuning

In [ ]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 10  0]
}

pipeline = Pipeline([
    ('model', SVC(probability=True))
])

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score),
    'roc_auc': make_scorer(roc_auc_score)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',
    cv=cv,
    return_train_score=True,
    n_jobs=-1
)

In [ ]:
mean_adoption_speed = important_df['AdoptionSpeed'].mean()
print(f"mean_adoption_speed: {mean_adoption_speed:.4f}")

important_df['AdoptionSpeedBinary'] = (important_df['AdoptionSpeed'] > mean_adoption_speed).astype(int)

status_counts = important_df['AdoptionSpeedBinary'].value_counts()

print("Count of rows where AdoptionSpeedBinary is 1:", status_counts.get(1, 0))
print("Count of rows where AdoptionSpeedBinary is 0:", status_counts.get(0, 0))

X = important_df.drop(columns=['AdoptionSpeedBinary']).drop(columns=['AdoptionSpeed'])
y = important_df['AdoptionSpeedBinary']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

grid.fit(X_train, y_train)

print("The best hyperparams:")
print(grid.best_params_)
print(f"The best ROC AUC: {grid.best_score_:.4f}")

results_df = pd.DataFrame(grid.cv_results_)
print("\nTop results (ROC AUC):")
print(results_df.sort_values(by='mean_test_roc_auc', ascending=False)[[
    'param_model__C', 'mean_test_accuracy', 'mean_test_precision', 'mean_test_recall',
    'mean_test_f1', 'mean_test_roc_auc'
]].head(20))

mean_adoption_speed: 2.5164
Count of rows where AdoptionSpeedBinary is 1: 7456
Count of rows where AdoptionSpeedBinary is 0: 7537
The best hyperparams:
{'model__var_smoothing': np.float64(1e-12)}
The best ROC AUC: 0.9976

Top results (ROC AUC):
   param_model__var_smoothing  mean_test_accuracy  mean_test_precision  \
0                1.000000e-12            0.997554             0.995404   
1                1.000000e-11            0.997554             0.995404   
2                1.000000e-10            0.997554             0.995404   
3                1.000000e-09            0.997554             0.995404   
4                1.000000e-08            0.997554             0.995404   
5                1.000000e-07            0.997554             0.995404   
6                1.000000e-06            0.997258             0.994813   

   mean_test_recall  mean_test_f1  mean_test_roc_auc  
0          0.999702      0.997547           0.997566  
1          0.999702      0.997547           0.997566

### Saving model

In [ ]:
best_model = grid.best_estimator_
joblib.dump(best_model, 'best_svm_model.pkl')

['best_naive_bayes_model.pkl']